# Distributions, z-scores and outliers

Core, Part 1 (Statistics I), lesson C3. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** describe a distribution by shape, center and spread, compute and read a z-score, use the empirical rule as orientation and not as a test, investigate an outlier before deciding to keep it, and compare two groups on center, spread and shape together.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each completed order, with its total in dollars: quantity × price − discount, summed over the order's lines.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(amount=order_items["quantity"] * order_items["product_id"].map(price) - order_items["discount_usd"])
order_total = lines.groupby("order_id")["amount"].sum().rename("total")
sales = orders[orders["status"] == "completed"].merge(order_total, on="order_id", validate="one_to_one")
totals = sales["total"]
assert len(sales) == 1799  # the canceled order of 5.00 is left out
# The sales of the whole chain on each day (181 days), and of each store on each day it was open (453 store-days).
sales["day"] = sales["ordered_at"].str[:10]
daily = sales.groupby("day")["total"].sum()
store_days = sales.groupby(["day", "store_id"], as_index=False)["total"].sum().rename(columns={"total": "sales"})
store_days["weekend"] = pd.to_datetime(store_days["day"]).dt.day_name().isin(["Saturday", "Sunday"])
assert (len(daily), len(store_days)) == (181, 453)
print("1,799 completed orders in `sales`, 181 days in `daily`, 453 store-days in `store_days`.")

## Unit 1: The shape of a column

**Predict first.** count the orders in 4-dollar slices (0 to 4, 4 to 8 and so on, the last slice 40 and over). Which slice will hold the most orders, and is the longer tail on the left or on the right? Will the skew still be that large without the order of 197.00?

In [ ]:
edges = [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 200]
slices = pd.cut(totals, edges, right=False).value_counts(sort=False)
print(slices.tolist())
print(round(totals.skew(), 2), round(totals.drop(totals.idxmax()).skew(), 2))
fig, ax = plt.subplots()
ax.hist(totals[totals < 40], bins=edges[:-1])
ax.set_title("Order totals under 40 dollars")
ax.set_xlabel("Order total (USD)")
ax.set_ylabel("Number of orders")

## Unit 2: The z-score

**Predict first.** how many standard deviations above the mean is each of the three largest orders? How many of the 1,799 orders have a z-score above 3 or below minus 3?

In [ ]:
z = (totals - totals.mean()) / totals.std()
for row in sales.assign(z=z).nlargest(3, "total").itertuples():
    print(row.order_id, row.store_id, row.total, round(row.z, 2))
print((z.abs() > 3).sum())

## Unit 3: The empirical rule, for orientation

**Predict first.** for a normal curve about 68, 95 and 99.7 percent of the values lie within 1, 2 and 3 standard deviations of the mean. Which column will match these shares better, the order totals or the daily sales of the chain?

In [ ]:
def share_within(x):
    zx = (x - x.mean()) / x.std()
    return [round(float((zx.abs() <= k).mean()), 3) for k in (1, 2, 3)]

print(share_within(totals))
print(share_within(daily))
print([round(float(stats.norm.cdf(k) - stats.norm.cdf(-k)), 3) for k in (1, 2, 3)])

## Unit 4: Investigate before you delete

**Predict first.** one order has a total above 190. Which line makes it so large, and what is the largest quantity on any other line of the whole table?

In [ ]:
suspect = int(sales.loc[sales["total"].idxmax(), "order_id"])
detail = lines[lines["order_id"] == suspect].merge(products[["product_id", "product_name"]], on="product_id", validate="many_to_one")
for row in detail.itertuples():
    print(row.product_name, row.quantity, row.amount)
print(lines[lines["order_id"] != suspect]["quantity"].max())

## Unit 5: Compare two groups on all three

**Predict first.** compare card and mobile orders. Do they differ in center, in spread or in shape?

In [ ]:
for method in ["card", "mobile"]:
    t = sales.loc[sales["payment_method"] == method, "total"]
    print(method, len(t), t.median(), t.quantile(0.75) - t.quantile(0.25), round(t.skew(), 1))

## Finish it

### Exercise 1: Finish it

**Task:** How many standard deviations from the mean is the sales total of Campus (store 2) on 2026-06-12? Use the mean and the standard deviation of all 453 store-day sales in `store_days`, with the standard deviation of `pandas`. Put the z-score in `answer_1`, rounded to 2 decimals.

**Expected:** `answer_1` is a number.

In [ ]:
campus_day = store_days[(store_days["store_id"] == 2) & (store_days["day"] == "2026-06-12")]["sales"].iloc[0]  # given: the sales of that store-day
# compute the z-score of campus_day among all store-day sales, and set answer_1
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 2.89, traps={73.53: "That is the distance from the mean in dollars. Divide it by the standard deviation to count standard deviations."})

<details>
<summary>Hint</summary>

A z-score is the distance from the mean divided by the standard deviation.

</details>

<details>
<summary>Solution</summary>

```python
campus_day = store_days[(store_days["store_id"] == 2) & (store_days["day"] == "2026-06-12")]["sales"].iloc[0]
answer_1 = round(float((campus_day - store_days["sales"].mean()) / store_days["sales"].std()), 2)
```

2.89: Campus sold 123.75 dollars that day, 73.53 above the mean of 50.22, and the standard deviation is 25.45. The check cell reports 73.53, the distance in dollars, if you forget to divide.

</details>

## On your own

### Exercise 2: On your own

**Task:** How do weekday store-days and weekend store-days differ? Before you start, say in one sentence what the result looks like: six numbers. In `store_days`, for the rows with `weekend` False and then True, find the median of `sales`, the IQR of `sales`, and the skew of `sales`. Put them in `answer_2` in this order: median of weekdays, median of weekends, IQR of weekdays, IQR of weekends, skew of weekdays, skew of weekends. Round each to 1 decimal. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of six numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [42.0, 64.1, 26.5, 38.9, 2.3, 0.4], traps={(44.5, 64.5, 22.9, 25.8, 0.0, 0.0): "Those are means and standard deviations. The task asks for the median, the IQR and the skew."})

<details>
<summary>Hint</summary>

Split `store_days` on `weekend` first. For each part, `quantile` gives the quartiles for the IQR and `skew` gives the skew.

</details>

<details>
<summary>Solution</summary>

```python
def three(x):
    return [x.median(), x.quantile(0.75) - x.quantile(0.25), x.skew()]

wd = three(store_days.loc[~store_days["weekend"], "sales"])
we = three(store_days.loc[store_days["weekend"], "sales"])
answer_2 = [round(float(v), 1) for v in (wd[0], we[0], wd[1], we[1], wd[2], we[2])]
```

Weekends sell more (median 64.1 against 42.0), vary more (IQR 38.9 against 26.5), and the weekday sales have the longer right tail (skew 2.3 against 0.4), which comes from the one extra-zero day, May 1; without that store-day the weekday skew is about 0.7. The three differences belong together in one description.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague removes every order with a z-score above 3 or below minus 3 and does not look at them. Look at the lines of those orders instead: keep an order unless one of its lines has a quantity above 3, the largest quantity of any other line in the table. Set `answer_3` to the number of flagged orders you would remove and the number you would keep, in that order.

**Expected:** `answer_3` is a list of two whole numbers.

In [ ]:
z_all = (totals - totals.mean()) / totals.std()
flagged = sales.loc[z_all.abs() > 3, "order_id"]  # the orders the colleague removes
answer_3 = [len(flagged), 0]  # the colleague removes all of them and keeps none of the flagged ones

In [ ]:
_ = check("Exercise 3", "answer_3", [1, 8], traps={(9, 0): "That removes every flagged order without looking at its lines. How many of them have a line with a quantity above 3?"})

<details>
<summary>Hint</summary>

Take the lines of the flagged orders with `isin`, and find the largest quantity of each order with `groupby`.

</details>

<details>
<summary>Solution</summary>

```python
z_all = (totals - totals.mean()) / totals.std()
flagged = sales.loc[z_all.abs() > 3, "order_id"]
biggest_line = lines[lines["order_id"].isin(flagged)].groupby("order_id")["quantity"].max()
remove = int((biggest_line > 3).sum())
answer_3 = [remove, len(flagged) - remove]
```

Nine orders are flagged. One has a line of 20 units; the other eight are orders of three lines with at most 3 units each, mostly the dearest items, which a real customer can order. Remove one, keep eight. The check cell reports [9, 0] for the colleague's rule, which also removes eight real orders.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")